# 10b — Containers, configuration, and deployment — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/10.md) · [Course map](../PLAN.md) · [Project](../../projects/stage10/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Explain image/container/network/volume boundaries
- Validate runtime configuration
- Design release and rollback evidence


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

An image is a filesystem and runtime specification; a container is a running instance. A volume retains data beyond a container’s lifecycle. A network connects services by service name; localhost inside a container refers to that container. Bind only intended interfaces and ports. A reverse proxy terminates HTTPS/TLS and routes requests; DNS resolves names.

Keep configuration outside the image. Secrets belong in a secret store or protected runtime environment, never source control or image layers. Use a non-root user, a small build context, dependency locking, and scanned/pinned release artifacts. Development version ranges are not a release lock.

CI should run tests, evaluation gates, and image builds. Deployment should apply tested migrations, check readiness, and support rollback. Rolling back an image cannot undo an incompatible database migration. In Coolify or another platform, configure persistent volumes, private networks, TLS, health checks, and backups deliberately. Local Docker success is one checkpoint, not evidence of operational readiness.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
config = {"host": "database", "port": 5432}
print("Containers connect using service DNS:", config)


## Exercise 1: Required configuration

Read a mapping with DATABASE_URL and API_KEY; reject missing/blank values or API_KEY shorter than 24 characters.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def validate_config(env):
    raise NotImplementedError


In [ ]:
assert validate_config({"DATABASE_URL": "postgresql://db/example", "API_KEY": "x"*32})["API_KEY"] == "x"*32
expect_error(ValueError, lambda: validate_config({}))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Fail at startup rather than using an insecure default.

</details>


## Exercise 2: Release gate

Return True only when tests, evals, migration_rehearsal, and restore_drill are all exactly True.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def release_ready(evidence):
    raise NotImplementedError


In [ ]:
assert not release_ready({"tests": True, "evals": True})
assert release_ready(dict.fromkeys(["tests", "evals", "migration_rehearsal", "restore_drill"], True))
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Missing evidence is not success.

</details>


## Exercise 3: Schema compatibility

A service supports schema versions inclusive [minimum,maximum]. Return whether current is compatible; reject inverted bounds.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def compatible(current, minimum, maximum):
    raise NotImplementedError


In [ ]:
assert compatible(3, 2, 4)
assert not compatible(5, 2, 4)
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Consider both upgrade and rollback versions.

</details>


## Independent transfer

Use the provided Dockerfile/Compose, run API+PostgreSQL, test persistence after restart, and rehearse backup/restore into a separate database. Add CI and document a Coolify deployment and rollback without publishing secrets.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why is localhost often wrong in Compose?

2. Why pin release artifacts?


## Reading and review

- [Primary reference 1](https://docs.docker.com/get-started/)
- [Primary reference 2](https://docs.docker.com/compose/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
